#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [23]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

You are already connected to a glueetl session f2b53f12-b68a-46b0-b209-ead5f4ce991d.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is 2880 minutes.
idle_timeout has been set to 2880 minutes.


You are already connected to a glueetl session f2b53f12-b68a-46b0-b209-ead5f4ce991d.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 4.0


You are already connected to a glueetl session f2b53f12-b68a-46b0-b209-ead5f4ce991d.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: G.1X
Setting new worker type to: G.1X


You are already connected to a glueetl session f2b53f12-b68a-46b0-b209-ead5f4ce991d.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: 2
Setting new number of workers to: 2



#### Cargar Dataset Raw


In [22]:
dyf = glueContext.create_dynamic_frame_from_catalog(
    database='lds_raw',
    table_name='suministro'
)

df = dyf.toDF()
df.show(5)
df.printSchema()


+-------------+----------+------------+--------------------+-------------+-----------------+---------------------+-----------------+
|id_suministro|id_cliente|id_ubicacion|direccion_suministro|nivel_tension|id_sist_electrico|fecha_alta_suministro|estado_suministro|
+-------------+----------+------------+--------------------+-------------+-----------------+---------------------+-----------------+
|      1000000|         1|          40| Pasaje Bolívar 2312|           BT|                5|  2017-05-04 09:04:18|         INACTIVO|
|      1000001|         2|          31| Pasaje Bolívar 2692|           BT|                8|  2000-09-18 08:33:09|           ACTIVO|
|      1000002|         3|           9|   Psje. El Sol 2734|           BT|                1|  2011-07-23 16:32:22|           ACTIVO|
|      1000003|         4|          43|Calle Los Pinos 7085|           BT|                1|  2006-03-07 01:47:14|           ACTIVO|
|      1000004|         5|          15|Psje. Venezuela 6696|         

#### Verificar número de filas


In [24]:
df.count()

12160


### Nulos por columna

In [26]:
import pyspark.sql.functions as F

df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-------------+----------+------------+--------------------+-------------+-----------------+---------------------+-----------------+
|id_suministro|id_cliente|id_ubicacion|direccion_suministro|nivel_tension|id_sist_electrico|fecha_alta_suministro|estado_suministro|
+-------------+----------+------------+--------------------+-------------+-----------------+---------------------+-----------------+
|            0|         0|           0|                   0|            0|                0|                    0|                0|
+-------------+----------+------------+--------------------+-------------+-----------------+---------------------+-----------------+


### Blancos por columna (solo strings)

In [27]:
df.select([
    F.count(F.when(F.col(c) == "", c)).alias(c)
    for c in df.columns if df.schema[c].dataType.simpleString() == "string"
]).show()


+--------------------+-------------+---------------------+-----------------+
|direccion_suministro|nivel_tension|fecha_alta_suministro|estado_suministro|
+--------------------+-------------+---------------------+-----------------+
|                   0|            0|                    0|                0|
+--------------------+-------------+---------------------+-----------------+


### Conteo por estado del suministro

In [28]:
df.groupBy("estado_suministro").count().show()


+-----------------+-----+
|estado_suministro|count|
+-----------------+-----+
|       SUSPENDIDO|  242|
|           ACTIVO|11657|
|         INACTIVO|  261|
+-----------------+-----+


### Chequear rango de fechas de alta

In [29]:
df.select(
    F.min("fecha_alta_suministro"),
    F.max("fecha_alta_suministro")
).show()


+--------------------------+--------------------------+
|min(fecha_alta_suministro)|max(fecha_alta_suministro)|
+--------------------------+--------------------------+
|       2000-01-01 06:21:09|       2024-12-30 00:00:00|
+--------------------------+--------------------------+
